   # Project 8 – Where are airports in Canada? (OSS4008A)
   Name: Nifemi Ogunmeru
   
   **Phase I, step 1:** load and clean OurAirports data, convert to an Earth Engine FeatureCollection, show an interactive map.
   
   **Data:** OurAirports (https://ourairports.com/data/), public domain.
   
   **How to run:** replace `YOUR_PROJECT_ID` in Cell 2, then Runtime → Run all.

In [ ]:
# Cell 1 – Load the libraries
import ee            # Earth Engine Python API
import geemap        # interactive Earth Engine maps (pre-installed in Colab)
import pandas as pd  # for working with tables

print('ee version:', ee.__version__, '| geemap version:', geemap.__version__)

ee version: 1.7.39 | geemap version: 0.38.8


In [ ]:
# Cell 2 – Log in to Earth Engine
ee.Authenticate()
ee.Initialize(project='YOUR_PROJECT_ID')

# Tiny test: ask Earth Engine's servers to send back a word
print('Earth Engine says:', ee.String('hello').getInfo())

Earth Engine says: hello


In [ ]:
# Cell 3 – Download the OurAirports data
# Source: OurAirports open data (https://ourairports.com/data/), public domain.
# We read the copy the OurAirports project keeps on GitHub (updated nightly).
URL = 'https://raw.githubusercontent.com/davidmegginson/ourairports-data/main/airports.csv'

# Read the whole CSV into a table.
# low_memory=False makes pandas read the file in one go, so each column gets one consistent type.
raw = pd.read_csv(URL, low_memory=False)

print('Downloaded on:', pd.Timestamp.now(tz='America/Toronto').date())  # Ottawa time, not UTC
print('Rows worldwide:', len(raw))
print('Columns:', raw.columns.tolist())
raw.head(3)                                          # show the first 3 rows

Downloaded on: 2026-10-18
Rows worldwide: 86258
Columns: ['id', 'ident', 'type', 'name', 'latitude_deg', 'longitude_deg', 'elevation_ft', 'continent', 'iso_country', 'iso_region', 'municipality', 'scheduled_service', 'icao_code', 'iata_code', 'gps_code', 'local_code', 'home_link', 'wikipedia_link', 'keywords']


,id,ident,type,name,latitude_deg,longitude_deg,elevation_ft,continent,iso_country,iso_region,municipality,scheduled_service,icao_code,iata_code,gps_code,local_code,home_link,wikipedia_link,keywords
0,6523,00A,heliport,Total RF Heliport,40.070985,-74.933689,11.0,NaN,US,US-PA,Bensalem,no,NaN,NaN,K00A,00A,https://www.penndot.pa.gov/TravelInPA/airports...,NaN,NaN
1,323361,00AA,small_airport,Aero B Ranch Airport,38.704022,-101.473911,3435.0,NaN,US,US-KS,Leoti,no,NaN,NaN,00AA,00AA,NaN,NaN,NaN
2,6524,00AK,small_airport,Lowell Field,59.947733,-151.692524,450.0,NaN,US,US-AK,Anchor Point,no,NaN,NaN,00AK,00AK,NaN,NaN,NaN


In [ ]:
# Cell 4a – Keep Canada and the 5 active facility types

# 1) Keep rows where the country code is 'CA' (Canada).
#    .copy() makes a separate table, so later edits don't affect 'raw'.
ca = raw[raw['iso_country'] == 'CA'].copy()
print('Canadian rows (all types):', len(ca))
print(ca['type'].value_counts(), '\n')    # count of each type BEFORE filtering

# 2) Keep only the 5 active types (this drops 'closed' and 'balloonport')
keep_types = ['large_airport', 'medium_airport', 'small_airport', 'heliport', 'seaplane_base']
ca = ca[ca['type'].isin(keep_types)]
print('After keeping the 5 active types:', len(ca))

Canadian rows (all types): 3369
type
small_airport     1254
closed             889
heliport           525
seaplane_base      450
medium_airport     233
large_airport       17
balloonport          1
Name: count, dtype: int64 

After keeping the 5 active types: 2479


In [ ]:
# Cell 4b – Show all 'large' airports so we can check them by eye
large = ca[ca['type'] == 'large_airport']
large[['ident', 'name', 'iata_code', 'iso_region', 'municipality']]

,ident,name,iata_code,iso_region,municipality
17994,CA-1291,(Duplicate)YEG,NaN,CA-AB,Edmonton
17995,CA-1292,YEG,NaN,CA-AB,Edmonton
21202,CYEG,Edmonton International Airport,YEG,CA-AB,Edmonton
21250,CYHM,John C. Munro Hamilton International Airport,YHM,CA-ON,Hamilton
21259,CYHZ,Halifax / Stanfield International Airport,YHZ,CA-NS,Halifax
21301,CYLW,Kelowna International Airport,YLW,CA-BC,Kelowna
21335,CYOW,Ottawa Macdonald-Cartier International Airport,YOW,CA-ON,Ottawa
21360,CYQB,Quebec Jean Lesage International Airport,YQB,CA-QC,Quebec
21363,CYQG,Windsor International Airport,YQG,CA-ON,Windsor
21425,CYUL,Montreal / Pierre Elliott Trudeau Internationa...,YUL,CA-QC,Montréal


In [ ]:
# Cell 4c – Remove junk rows, keep useful columns, check coordinates

# 1) Two rules that catch bad rows:
#    - the name contains "duplicate"
#    - a 'large_airport' with no IATA code (every real large Canadian airport has one)
is_duplicate  = ca['name'].str.contains('duplicate', case=False, na=False)
large_no_iata = (ca['type'] == 'large_airport') & ca['iata_code'].isna()
junk = is_duplicate | large_no_iata          # '|' means OR

# Show exactly what we remove, so a future data refresh can't silently drop a real airport
print('Removing', int(junk.sum()), 'junk rows:')
print(ca[junk][['ident', 'name', 'type', 'latitude_deg', 'longitude_deg']], '\n')
ca = ca[~junk]                               # '~' means NOT: keep everything that isn't junk

# 2) Keep only the columns we need, with shorter names
airports = ca[['ident', 'name', 'type', 'iata_code', 'icao_code', 'municipality',
               'iso_region', 'latitude_deg', 'longitude_deg', 'elevation_ft',
               'scheduled_service']].rename(columns={'latitude_deg': 'lat',
                                                     'longitude_deg': 'lon',
                                                     'iso_region': 'province'})
airports['province'] = airports['province'].str.replace('CA-', '', regex=False)  # 'CA-ON' -> 'ON'

# 3) Sanity check: keep only points inside a rough box around Canada
inside = airports['lat'].between(41, 84) & airports['lon'].between(-142, -52)
print('Points outside the Canada box:', int((~inside).sum()))
airports = airports[inside].reset_index(drop=True)

# 4) Final summary
print('Final count:', len(airports), '\n')
print(airports['type'].value_counts())

Removing 2 junk rows:
         ident            name           type  latitude_deg  longitude_deg
17994  CA-1291  (Duplicate)YEG  large_airport     53.309700    -113.580000
17995  CA-1292             YEG  large_airport     50.341652    -106.699219 

Points outside the Canada box: 0
Final count: 2477 

type
small_airport     1254
heliport           525
seaplane_base      450
medium_airport     233
large_airport       15
Name: count, dtype: int64


In [ ]:
# Cell 5a – Turn the table into an Earth Engine FeatureCollection

# Earth Engine can't store NaN, so replace missing values with empty text ''
airports_ee = airports.copy()
airports_ee['elevation_ft'] = airports_ee['elevation_ft'].round()       # whole feet
airports_ee = airports_ee.astype(object).where(airports_ee.notna(), '')

# Build one ee.Feature per airport: geometry = point, properties = everything else
features = []
for row in airports_ee.to_dict('records'):             # each row as a dictionary
    lon, lat = row.pop('lon'), row.pop('lat')          # take the coordinates out of the properties
    point = ee.Geometry.Point([lon, lat])              # NOTE: longitude comes FIRST in Earth Engine
    features.append(ee.Feature(point, row))

airports_fc = ee.FeatureCollection(features)

# Check: the count should match the table, and one feature should look sensible
print('Features in the collection:', airports_fc.size().getInfo())
print('First feature:', airports_fc.first().getInfo()['properties'])

Features in the collection: 2477
First feature: {'elevation_ft': '', 'iata_code': '', 'icao_code': '', 'ident': 'CA-0002', 'municipality': 'Saint-Lazare', 'name': 'Pourvoirie Joncas Airstrip', 'province': 'QC', 'scheduled_service': 'no', 'type': 'small_airport'}


In [ ]:
# Cell 5b – Interactive map: one layer per airport type, plus a legend

# Colour (hex, no '#') and legend label for each type
styles = {
    'large_airport':  {'color': 'd7191c', 'label': 'Large airports'},
    'medium_airport': {'color': 'fdae61', 'label': 'Medium airports'},
    'small_airport':  {'color': '2b83ba', 'label': 'Small airports'},
    'seaplane_base':  {'color': '5ab4ac', 'label': 'Seaplane bases'},
    'heliport':       {'color': '7b3294', 'label': 'Heliports'},
}

# Map centred on Canada. NOTE: geemap's center is [lat, lon], unlike ee.Geometry.Point([lon, lat])
Map = geemap.Map(center=[56, -96], zoom=4)

# Add layers in reverse order (heliports first, large airports last),
# because layers added later are drawn on top
for t, s in reversed(list(styles.items())):
    layer = airports_fc.filter(ee.Filter.eq('type', t))     # only airports of this type
    Map.addLayer(layer, {'color': s['color']}, s['label'],
                 shown=(t not in ['heliport', 'seaplane_base']))  # start with these 2 hidden

# Legend: label -> colour (the legend wants a '#' in front of the colour)
Map.add_legend(title='Airport types',
               legend_dict={s['label']: '#' + s['color'] for s in styles.values()})

Map   # display the map

Map(center=[56, -96], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', tra…

In [ ]:
# Cell 6 – Save the cleaned table (goes in the data/ folder on GitHub)
airports.to_csv('canada_airports_clean.csv', index=False)   # index=False: don't save pandas' row numbers
print('Saved', len(airports), 'rows to canada_airports_clean.csv')

# Download the file to your computer
from google.colab import files
files.download('canada_airports_clean.csv')

Saved 2477 rows to canada_airports_clean.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>